# Lab: Checkpoint & Store for Memory

This notebook demonstrates **why memory matters** in conversational agents and how
LangGraph's **SQLite checkpointer** solves the problem.

## Structure

| Section | What you will see |
|---------|-------------------|
| **Shared Setup** | Load LLM (same pattern as the search-tools exercise) |
| **Round 1 — No Memory** | Agent forgets everything between calls |
| **Round 2 — With SQLite Checkpoint** | You fill in the TODOs and watch the agent remember |

> **Goal:** understand that without a checkpointer each `invoke` is stateless;
> adding `SqliteSaver` gives the graph persistent, per-thread memory.

## Shared Setup (Run Once)

Instructions:
1. Set `MODEL_PROVIDER = "openai"` or `"gemini"` below.
2. Run this cell before anything else — all other cells depend on `llm`.
3. If an import fails, run `uv add langchain-openai` or `uv add langchain-google-genai` in your terminal and restart the kernel.
4. The `langgraph-checkpoint-sqlite` package is required for Round 2. If missing, run `uv add langgraph-checkpoint-sqlite`.

In [5]:
import os
import logging

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# ── Model selection ───────────────────────────────────────────────────────────
MODEL_PROVIDER = "openai"  # "gemini" | "openai"

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# ── Shared LLM ────────────────────────────────────────────────────────────────
if MODEL_PROVIDER == "openai":
    assert OPENAI_API_KEY, "OPENAI_API_KEY not found — check your .env file"
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=OPENAI_API_KEY,
        temperature=0,
    )
else:
    assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash-lite",
        google_api_key=GEMINI_API_KEY,
        temperature=0,
    )

logger.info("Setup complete — LLM provider: %s", MODEL_PROVIDER)

INFO | Setup complete — LLM provider: openai


---
## Round 1 — No Memory

We build a **simple chatbot graph** using `StateGraph` + `MessagesState` — no tools, no
`create_react_agent`.

The graph has a single node (`chatbot`) that calls the LLM and returns its reply.
We compile it **without** a checkpointer.

```
START → chatbot → END
```

### What to observe
- **Invoke 1**: tell the agent your favourite food.  
- **Invoke 2**: ask the agent what your favourite food is.  

Because each `invoke` starts with a fresh state (no persisted history), the agent
will **not** remember what you told it in Invoke 1.

In [6]:
# ── Build the chatbot graph (no checkpointer) ─────────────────────────────────

def chatbot_node(state: MessagesState):
    """Single node: send all messages to the LLM and append its reply."""
    logger.debug("chatbot_node called with %d message(s)", len(state["messages"]))
    response = llm.invoke(state["messages"])
    logger.info("LLM response: %s", response.content)
    return {"messages": [response]}


graph_builder = StateGraph(MessagesState)
graph_builder.add_node("chatbot", chatbot_node)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

# No checkpointer → every invoke is completely stateless
graph_no_memory = graph_builder.compile()

logger.info("Graph compiled (no memory)")

INFO | Graph compiled (no memory)


In [7]:
# ── Invoke 1: Tell the agent your favourite food ───────────────────────────────

result_1 = graph_no_memory.invoke(
    {"messages": [HumanMessage(content="Hi! My favourite food is sushi. Please remember that.")]}
)

print("=== Invoke 1 — Agent reply ===")
print(result_1["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | LLM response: Got it! Your favorite food is sushi. If you have any questions or want to talk more about sushi, feel free to ask!


=== Invoke 1 — Agent reply ===
Got it! Your favorite food is sushi. If you have any questions or want to talk more about sushi, feel free to ask!


In [8]:
# ── Invoke 2: Ask about the favourite food ─────────────────────────────────────
# Each invoke creates a BRAND NEW state — the agent has no knowledge of Invoke 1.

result_2 = graph_no_memory.invoke(
    {"messages": [HumanMessage(content="What is my favourite food?")]}
)

print("=== Invoke 2 — Agent reply (expect: no memory) ===")
print(result_2["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | LLM response: I don't have access to personal information about you, so I can't know your favorite food. However, if you tell me what you enjoy, I can help you with recipes or suggestions!


=== Invoke 2 — Agent reply (expect: no memory) ===
I don't have access to personal information about you, so I can't know your favorite food. However, if you tell me what you enjoy, I can help you with recipes or suggestions!


**Expected result:** the agent says something like *"I don't have access to personal information about you"*.
It has no idea about sushi because the first conversation was never saved.

---
## Round 2 — With SQLite Checkpoint (Memory Enabled)

LangGraph's **checkpointer** serialises the full graph state after every step and
stores it. When you invoke again with the **same `thread_id`**, LangGraph loads that
saved state first — so the agent sees the entire conversation history.

We use `SqliteSaver` from `langgraph-checkpoint-sqlite`:

```python
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(conn)
graph = graph_builder.compile(checkpointer=memory)
```

Each conversation is identified by a **`thread_id`** passed inside `config`:

```python
config = {"configurable": {"thread_id": "alice"}}
graph.invoke({...}, config=config)
```

Different `thread_id` values → independent conversation histories.

### Your task
Fill in every `# TODO` below and re-run the Invoke cells to see the agent remember.

In [9]:
# ── TODO: Import SqliteSaver and compile the graph WITH memory ─────────────────

# Step 1 — imports
# TODO: import sqlite3
import sqlite3
# TODO: from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.sqlite import SqliteSaver

# Step 2 — create an in-memory SQLite connection and wrap it with SqliteSaver
# check_same_thread=False is required so Jupyter can use the connection across cells
# TODO: conn = sqlite3.connect(":memory:", check_same_thread=False)
conn = sqlite3.connect(":memory:", check_same_thread=False)
# TODO: memory = SqliteSaver(conn)
memory = SqliteSaver(conn)

# Step 3 — compile the SAME graph_builder but pass checkpointer=memory
# Note: we reuse graph_builder defined in Round 1 — no need to redefine nodes.
# TODO: graph_with_memory = graph_builder.compile(checkpointer=memory)
graph_with_memory = graph_builder.compile(checkpointer=memory)

# TODO: logger.info("Graph compiled (with SQLite memory)")
logger.info("Graph compiled (with SQLite memory)")

INFO | Graph compiled (with SQLite memory)


In [20]:
# ── TODO: Define a config with a thread_id ─────────────────────────────────────
# The thread_id groups messages into one conversation thread.
# Any string works — think of it as a conversation/session ID.

# TODO: config = {"configurable": {"thread_id": "user_alice"}}
config = {"configurable": {"thread_id": "user_bob"}}

In [21]:
# ── TODO: Invoke 1 (with memory) — tell the agent your favourite food ──────────

# TODO: result_m1 = graph_with_memory.invoke(
#     {"messages": [HumanMessage(content="Hi! My favourite food is sushi. Please remember that.")]},
#     config=config,
# )
result_m1 = graph_with_memory.invoke(
    {"messages": [HumanMessage(content="Hi! My favourite food is sushi. Please remember that.")]},
    config=config,
)

# TODO: print("=== Invoke 1 (memory) — Agent reply ===")
print("=== Invoke 1 (memory) — Agent reply ===")
# TODO: print(result_m1["messages"][-1].content)
print(result_m1["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | LLM response: Got it! Your favorite food is sushi. If you have any questions or want to talk more about sushi, feel free to ask!


=== Invoke 1 (memory) — Agent reply ===
Got it! Your favorite food is sushi. If you have any questions or want to talk more about sushi, feel free to ask!


In [22]:
# ── TODO: Invoke 2 (with memory) — ask about the favourite food ───────────────
# Use the SAME config (same thread_id) so LangGraph loads the saved state.

# TODO: result_m2 = graph_with_memory.invoke(
#     {"messages": [HumanMessage(content="What is my favourite food?")]},
#     config=config,
# )
result_m2 = graph_with_memory.invoke(
    {"messages": [HumanMessage(content="What is my favourite food?")]},
    config=config,
)

# TODO: print("=== Invoke 2 (memory) — Agent reply (expect: sushi!) ===")
print("=== Invoke 2 (memory) — Agent reply (expect: sushi!) ===")
# TODO: print(result_m2["messages"][-1].content)
print(result_m2["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | LLM response: Your favorite food is sushi!


=== Invoke 2 (memory) — Agent reply (expect: sushi!) ===
Your favorite food is sushi!


In [23]:
print(result_m2["messages"])

[HumanMessage(content='Hi! My favourite food is sushi. Please remember that.', additional_kwargs={}, response_metadata={}, id='5fd252e2-0619-4779-a47b-f0ac3402a859'), AIMessage(content='Got it! Your favorite food is sushi. If you have any questions or want to talk more about sushi, feel free to ask!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 19, 'total_tokens': 46, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DO2RtNHS58ZgdIdyLDZu5idZ3Zlr5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2fb1-85e2-7362-b614-80eebb66f39f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 

**Expected result:** the agent replies with *"Your favourite food is sushi"* (or similar).
LangGraph reloaded the thread's saved state before running the second invoke, so the
full message history — including your first message — was available to the LLM.

---
## Bonus: Try a different `thread_id`

Change the `thread_id` to `"user_bob"` and repeat the two invokes.
You will see that `"user_bob"` has no memory of sushi — threads are fully isolated.